In [17]:
import pandas as pd
from sklearn import datasets
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics

In [24]:
# 1. Wczytanie danych i czyszczenie
df = pd.read_csv("../output/features/all_features.csv")
df = df.dropna()

# Konwersja kolumny na tekst, aby uniknąć błędów
df["gbifID"] = df["gbifID"].astype(str)

# 2. Jeśli zawiera 'pseudo' -> 0, w przeciwnym wypadku -> 1
y = df["gbifID"].str.contains("pseudo", case=True).map({True: 0, False: 1})

columns_to_drop = ["gbifID", "decimalLongitude", "decimalLatitude"]
X = df.drop(columns=columns_to_drop)


In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, random_state=0)


In [26]:
#siatka parametrów do sprawdzenia
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 3, 4, None], 
    'min_samples_split': [2, 5, 10],     
    'max_features': [None, 'sqrt', 'log2'],
    'class_weight': [None, 'balanced']
}

#konfiguracja grid
grid_search = GridSearchCV(estimator=DecisionTreeClassifier(random_state=0), 
                           param_grid=param_grid, 
                           cv=5, 
                           n_jobs=5, 
                           verbose=1, 
                           scoring='accuracy')

# grid serach
grid_search.fit(X_train, y_train)

# najlepsze
print(f"Najlepsze parametry: {grid_search.best_params_}")
print(f"Najlepszy wynik (accuracy): {grid_search.best_score_}")

# zapis
best_tree = grid_search.best_estimator_

#Sprawdzenie na zbiorze testowym
y_pred = best_tree.predict(X_test)
print(metrics.classification_report(y_test, y_pred))

Fitting 5 folds for each of 144 candidates, totalling 720 fits


Najlepsze parametry: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2}
Najlepszy wynik (accuracy): 0.7903997185322487
              precision    recall  f1-score   support

           0       0.67      0.63      0.65       744
           1       0.83      0.86      0.84      1569

    accuracy                           0.78      2313
   macro avg       0.75      0.74      0.75      2313
weighted avg       0.78      0.78      0.78      2313

